# Threshold Strategy Diagnostic

После Sprint 1 (signal exists) и Sprint 2 (overfit fixed, train/val ratio 1.7×)
Sharpe всё ещё отрицательный (-2.89 best). Гипотеза: проблема в **стратегии
выбора threshold'а**, а не в модели — все calibrate-методы (ACI/DtACI/Bayes)
сходятся к ~0.42 (= mean prediction), что даёт 70% BUY-сигналов и тонет в
costs.

Этот ноутбук — **диагностический gate**. Если хоть одна комбинация
**model × strategy × horizon** даёт Sharpe > 0 на test, задача обучаема
и архитектурные спринты 4-5 имеют смысл. Если **все 32 ячейки красные** —
cost-aware labels слишком строгие, нужно менять задачу (горизонты, label
thresholds, фичи).

## Стратегии

| # | Strategy | Threshold формула |
|---|---|---|
| 1 | **prob_cutoff** | T = 0.5 (default) — текущий baseline |
| 2 | **top_5%** | T = 95-й перцентиль mean(pred) на val |
| 3 | **isotonic** | calibrated_mean = IsotonicRegression(val), T = 0.5 |
| 4 | **max_pnl** | T = argmax_T mean(lr_actual - cost \| pred > T) на val |

## Модели

1. **iTransformer** ConcurrentEnsemble (M=5) — наш основной стек после Sprint 2
2. **LightGBM** — простой baseline из Sprint 1, для проверки model-side

Время: ~12-15 минут на Colab L4 (training + LightGBM + 32 strategy backtest).

## 0. Окружение

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    if PROJECT_ROOT.exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH],
                       check=True, env=env)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard',
                        f'origin/{REPO_BRANCH}'], check=True, env=env)
    else:
        subprocess.run(
            ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
             f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(PROJECT_ROOT)],
            check=True, env=env,
        )
else:
    PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

In [ ]:
if IN_COLAB:
    subprocess.run(
        ['pip', 'install', '--quiet',
         'torch>=2.2', 'pandas>=2.1', 'pyarrow>=15.0',
         'scikit-learn>=1.4', 'lightgbm>=4.0',
         'tqdm>=4.66', 'matplotlib>=3.7'],
        check=True,
    )
    print('Готово.')

In [ ]:
import sys

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import dataclasses

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import lightgbm as lgb

from graduate_work.config import default_config

cfg = default_config()
# Тот же anti-collapse stack, что и в itransformer_ensemble_aci.ipynb.
cfg = dataclasses.replace(
    cfg,
    model=dataclasses.replace(
        cfg.model,
        architecture='itransformer',
        itransformer_seq_len=cfg.data.window_size,
    ),
    trading=dataclasses.replace(
        cfg.trading,
        loss_objective='composite',
        composite_inner_loss='asl',
        composite_sharpe_weight=0.0,
        use_class_balanced_pos_weight=True,
    ),
    training=dataclasses.replace(
        cfg.training,
        use_imbsam=True, mixup_alpha=0.2, mixup_p=0.5,
        ensemble_repulsion_weight=0.1,
        use_concurrent_ensemble=True,
    ),
)
print('Stack:', cfg.model.architecture,
      'tau=', cfg.model.logit_adjust_tau,
      'wd=', cfg.training.weight_decay,
      'patience=', cfg.training.early_stopping_patience)

## 1. Данные + features (тот же pipeline, что и в других ноутбуках)

In [ ]:
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    src_dir = DRIVE_BASE / 'data' / 'raw' / 'Algopack'
    if not src_dir.exists():
        src_dir = DRIVE_BASE / 'data' / 'raw'
    if src_dir.exists():
        shutil.copytree(src_dir, cfg.paths.data_raw, dirs_exist_ok=True)
        print(f'Скопировано из {src_dir}')

In [ ]:
from graduate_work.features.algopack_features import (
    obstats_features, orderstats_features, tradestats_features, order_to_trade_ratio,
)
from graduate_work.features.targets import cost_aware_classification_labels, lr_columns


def _load_supercandle(product: str, ticker: str) -> pd.DataFrame:
    p = cfg.paths.data_raw / 'algopack' / product / f'{ticker}.csv'
    if not p.exists():
        return pd.DataFrame()
    df = pd.read_csv(p)
    if df.empty or 'tradedate' not in df.columns:
        return df
    if 'tradetime' in df.columns:
        ts = pd.to_datetime(df['tradedate'].astype(str) + ' ' + df['tradetime'].astype(str),
                            utc=True, errors='coerce')
    else:
        ts = pd.to_datetime(df['tradedate'], utc=True, errors='coerce')
    df.index = ts; df.index.name = 'begin'
    return df.dropna(how='all').sort_index()


def _ts_to_ohlcv(ts: pd.DataFrame, ticker: str) -> pd.DataFrame:
    out = pd.DataFrame(index=ts.index)
    out['open'] = ts['pr_open'].astype(float)
    out['high'] = ts['pr_high'].astype(float)
    out['low']  = ts['pr_low'].astype(float)
    out['close']= ts['pr_close'].astype(float)
    out['volume'] = ts['vol'].astype(float)
    out['value']  = ts['val'].astype(float)
    out['vwap']   = ts['pr_vwap'].astype(float)
    out['ticker'] = ticker
    return out


def _build_log_returns(close: pd.Series) -> pd.DataFrame:
    lr = np.log(close / close.shift(1))
    return pd.DataFrame({
        'lr_1':  lr,
        'lr_5':  lr.rolling(5).sum(),
        'lr_15': lr.rolling(15).sum(),
        'lr_30': lr.rolling(30).sum(),
    }, index=close.index)


ts_data, os_data, ob_data = {}, {}, {}
for ticker in cfg.data.tickers:
    ts_data[ticker] = _load_supercandle('tradestats', ticker)
    os_data[ticker] = _load_supercandle('orderstats', ticker)
    ob_data[ticker] = _load_supercandle('obstats', ticker)

frames = []
for ticker in cfg.data.tickers:
    if ts_data[ticker].empty:
        continue
    feat = _ts_to_ohlcv(ts_data[ticker], ticker)
    feat = feat.join(_build_log_returns(feat['close']), how='left')
    feat = feat.join(tradestats_features(ts_data[ticker]), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(orderstats_features(os_data[ticker]), how='left')
    if not ob_data[ticker].empty:
        feat = feat.join(obstats_features(ob_data[ticker]), how='left')
    if not os_data[ticker].empty:
        feat = feat.join(order_to_trade_ratio(os_data[ticker], ts_data[ticker]), how='left')
    feat = feat.fillna(0.0)
    frames.append(feat)

full = pd.concat(frames).sort_index()

# Targets
out_parts = []
costs = cfg.trading.commission_rate + cfg.trading.slippage_rate
for ticker, sub in full.groupby('ticker'):
    labels = cost_aware_classification_labels(
        open_price=sub['open'], close_price=sub['close'],
        horizons=cfg.data.horizons,
        entry_cost=costs, exit_cost=costs,
        label_smoothing=0.0, direction='long',
    )
    out_parts.append(sub.join(labels, how='left'))
full_with_targets = pd.concat(out_parts).sort_values(['ticker']).sort_index(kind='stable')

target_cols = [f'target_h{h}' for h in cfg.data.horizons]
lr_cols = lr_columns(cfg.data.horizons)
feature_cols = [c for c in full_with_targets.columns
                if c not in ('ticker',) and c not in target_cols and c not in lr_cols]
print(f'frame: {full_with_targets.shape}, features: {len(feature_cols)}')

In [ ]:
from graduate_work.features.pipeline import chronological_split, _build_arrays
from graduate_work.features.scaler import StandardScaler
from graduate_work.strategy import extract_lr_array

train_df, val_df, test_df = chronological_split(
    full_with_targets,
    cfg.data.train_ratio, cfg.data.val_ratio,
    purge_horizon=max(cfg.data.horizons),
)
test_prices_raw = test_df[['open', 'close', 'ticker']].copy()

scaler = StandardScaler()
scaler.fit(train_df, feature_cols)
train_df_s = scaler.transform(train_df)
val_df_s = scaler.transform(val_df)
test_df_s = scaler.transform(test_df)

train = _build_arrays(train_df_s, feature_cols, target_cols, cfg.data.window_size, desc='train')
val   = _build_arrays(val_df_s,   feature_cols, target_cols, cfg.data.window_size, desc='val')
test  = _build_arrays(test_df_s,  feature_cols, target_cols, cfg.data.window_size, desc='test')

train_lr = extract_lr_array(train_df, train, cfg.data.horizons)
val_lr = extract_lr_array(val_df, val, cfg.data.horizons)
test_lr_array = extract_lr_array(test_df, test, cfg.data.horizons)  # для max_pnl на test
print(f'train: {train["x"].shape}, val: {val["x"].shape}, test: {test["x"].shape}')

## 2. Train iTransformer ensemble (тот же стек что в itransformer_ensemble_aci.ipynb)

Время на L4: ~9-12 минут (concurrent, M=5, patience=5 → ранний early stop).

In [ ]:
from graduate_work.model import build_model
from graduate_work.training import ConcurrentDeepEnsembleTrainer, ensemble_predict

INPUT_DIM = train['x'].shape[-1]
NUM_HORIZONS = len(cfg.data.horizons)

def model_factory(seed: int):
    return build_model(input_dim=INPUT_DIM, num_horizons=NUM_HORIZONS, cfg=cfg.model)

ckpt_dir = cfg.paths.checkpoints / 'threshold_strategy_run'
ensemble = ConcurrentDeepEnsembleTrainer(
    model_factory, cfg.training,
    ensemble_size=5,
    data_cfg=cfg.data, trading_cfg=cfg.trading,
    base_seed=cfg.training.seed,
    svgd_repulsion_weight=cfg.training.ensemble_repulsion_weight,
)
history = ensemble.fit(train, val, checkpoint_dir=ckpt_dir,
                       train_lr=train_lr, val_lr=val_lr)
print('\n=== iTransformer ensemble ready ===')
for i, (s, h) in enumerate(zip(history.seeds, history.member_histories)):
    print(f'  m{i:02d} seed={s}: best_val={h.best_val_loss:.5f} ep={h.best_epoch}')

In [ ]:
from graduate_work.strategy import build_predictions_frame, attach_actual_targets

val_mean_it, val_std_it = ensemble_predict(
    ensemble.members, val['x'],
    batch_size=cfg.training.batch_size, apply_sigmoid=True,
)
test_mean_it, test_std_it = ensemble_predict(
    ensemble.members, test['x'],
    batch_size=cfg.training.batch_size, apply_sigmoid=True,
)
val_predictions_it = build_predictions_frame(
    val['timestamp'], val['ticker'], val_mean_it, val_std_it, cfg.data.horizons,
)
test_predictions_it = build_predictions_frame(
    test['timestamp'], test['ticker'], test_mean_it, test_std_it, cfg.data.horizons,
)
val_targets = attach_actual_targets(val, cfg.data.horizons)
test_targets = attach_actual_targets(test, cfg.data.horizons)
print(f'val_pred: {val_predictions_it.shape}, test_pred: {test_predictions_it.shape}')

## 3. Train LightGBM baseline на тех же фичах

Last-bar features (без windowing). Совместимо с iTransformer'ом по
feature space — справедливое сравнение.

In [ ]:
def _tabular(df: pd.DataFrame, target_col: str) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    valid = df[target_col].notna()
    X = df.loc[valid, feature_cols].to_numpy(dtype=np.float32)
    y = df.loc[valid, target_col].to_numpy(dtype=np.float32)
    meta = df.loc[valid, ['ticker']].copy()
    # ВАЖНО: явно сохраняем UTC-aware timestamps. .values стирает tz,
    # потом merge с iTransformer-предсказаниями (UTC-aware) падает с
    # "datetime64[ns] vs datetime64[ns, UTC]".
    meta['timestamp'] = pd.to_datetime(df.loc[valid].index, utc=True)
    return X, y, meta

lgbm_models: dict[int, lgb.LGBMClassifier] = {}
lgbm_val_preds: list[pd.DataFrame] = []
lgbm_test_preds: list[pd.DataFrame] = []

for h, target_col in zip(cfg.data.horizons, target_cols):
    X_tr, y_tr, _ = _tabular(train_df, target_col)
    X_va, y_va, meta_va = _tabular(val_df, target_col)
    X_te, y_te, meta_te = _tabular(test_df, target_col)

    # DataFrame-обёртка с feature_cols сохраняет имена фичей в LightGBM,
    # иначе sklearn спамит UserWarning'ами при predict_proba.
    X_tr_df = pd.DataFrame(X_tr, columns=feature_cols)
    X_va_df = pd.DataFrame(X_va, columns=feature_cols)
    X_te_df = pd.DataFrame(X_te, columns=feature_cols)

    model = lgb.LGBMClassifier(
        n_estimators=200, num_leaves=31, learning_rate=0.05,
        min_child_samples=200, reg_lambda=1.0, reg_alpha=0.1,
        random_state=42, verbosity=-1, n_jobs=-1,
    )
    model.fit(X_tr_df, y_tr, eval_set=[(X_va_df, y_va)],
              callbacks=[lgb.early_stopping(20, verbose=False)])
    lgbm_models[h] = model

    p_va = model.predict_proba(X_va_df)[:, 1]
    p_te = model.predict_proba(X_te_df)[:, 1]

    val_df_h = pd.DataFrame({
        'timestamp': pd.to_datetime(meta_va['timestamp'].values, utc=True),
        'ticker': meta_va['ticker'].values,
        'horizon': int(h),
        'mean': p_va.astype(np.float32),
        'std': np.zeros_like(p_va, dtype=np.float32),
    })
    test_df_h = pd.DataFrame({
        'timestamp': pd.to_datetime(meta_te['timestamp'].values, utc=True),
        'ticker': meta_te['ticker'].values,
        'horizon': int(h),
        'mean': p_te.astype(np.float32),
        'std': np.zeros_like(p_te, dtype=np.float32),
    })
    lgbm_val_preds.append(val_df_h)
    lgbm_test_preds.append(test_df_h)
    print(f'  h={h}: best_iter={model.best_iteration_}, '
          f'mean_val_pred={p_va.mean():.4f}, mean_test_pred={p_te.mean():.4f}')

val_predictions_gbm = pd.concat(lgbm_val_preds, ignore_index=True)
test_predictions_gbm = pd.concat(lgbm_test_preds, ignore_index=True)
print(f'\nLGBM val_pred: {val_predictions_gbm.shape}, test_pred: {test_predictions_gbm.shape}')
print(f'val_pred timestamp dtype: {val_predictions_gbm["timestamp"].dtype}')

## 4. Подготовка lr-targets (для max_pnl threshold)

Нужно `actual = lr_h{h}` per (timestamp, ticker, horizon) на val.
Для iTransformer'а у нас уже есть `val_lr` (windowed). Для LightGBM
(table-form) — нужно собрать из `val_df`.

In [ ]:
from graduate_work.strategy.calibration import attach_lr_targets

# Long-form lr_targets для iTransformer val (uses windowed dataset).
val_lr_targets_it = attach_lr_targets(
    full_with_targets, val, cfg.data.horizons,
)

# Long-form lr_targets для LightGBM val (по valid строкам val_df, table-form).
lgbm_val_lr_rows = []
for h in cfg.data.horizons:
    target_col = f'target_h{h}'
    lr_col = f'lr_h{h}'
    sub = val_df[val_df[target_col].notna() & val_df[lr_col].notna()]
    for ts, row in sub.iterrows():
        lgbm_val_lr_rows.append({
            'timestamp': ts, 'ticker': row['ticker'],
            'horizon': int(h), 'actual': float(row[lr_col]),
        })
val_lr_targets_gbm = pd.DataFrame(lgbm_val_lr_rows)
# Принудительный UTC — иначе merge с UTC-aware predictions упадёт.
if not val_lr_targets_gbm.empty:
    val_lr_targets_gbm['timestamp'] = pd.to_datetime(
        val_lr_targets_gbm['timestamp'], utc=True,
    )
print(f'iTransformer val_lr: {val_lr_targets_it.shape}, '
      f'ts dtype={val_lr_targets_it["timestamp"].dtype}')
print(f'LightGBM val_lr: {val_lr_targets_gbm.shape}, '
      f'ts dtype={val_lr_targets_gbm["timestamp"].dtype if not val_lr_targets_gbm.empty else "empty"}')

## 5. Strategy comparison matrix

Per-horizon backtest. Каждая ячейка = одна стратегия применена к одному
горизонту одной модели → Sharpe + total_return.

In [ ]:
from graduate_work.backtest import compute_metrics, run_backtest
from graduate_work.strategy import (
    apply_isotonic_calibration, fit_isotonic_per_horizon,
    max_pnl_threshold, signals_per_horizon_threshold, top_k_threshold,
)

bars_per_year = cfg.data.bars_per_year
cost_per_trade = 2.0 * (cfg.trading.commission_rate + cfg.trading.slippage_rate)


def _backtest_signals(signals: pd.DataFrame) -> dict:
    n_buy = int((signals['action'] == 'BUY').sum())
    if n_buy == 0:
        return {'n_buy': 0, 'sharpe': 0.0, 'total_return': 0.0,
                'win_rate': 0.0, 'max_dd': 0.0}
    bt = run_backtest(signals, test_prices_raw, cfg.trading)
    m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
    return {
        'n_buy': n_buy, 'sharpe': float(m['sharpe']),
        'total_return': float(m['total_return']),
        'win_rate': float(m['win_rate']),
        'max_dd': float(m['max_drawdown']),
    }


def _evaluate_strategies(
    model_name: str,
    val_predictions: pd.DataFrame,
    test_predictions: pd.DataFrame,
    val_targets_local: pd.DataFrame,
    val_lr_targets_local: pd.DataFrame,
) -> list[dict]:
    rows = []
    iso_calibrators = fit_isotonic_per_horizon(val_predictions, val_targets_local)
    test_pred_calibrated = apply_isotonic_calibration(test_predictions, iso_calibrators)

    for h in cfg.data.horizons:
        h_val = val_predictions[val_predictions['horizon'] == h]
        h_val_calibrated = apply_isotonic_calibration(h_val, iso_calibrators)

        # 1. prob_cutoff @ 0.5 baseline
        sig_pc = signals_per_horizon_threshold(test_predictions, h, threshold=0.5)
        rows.append({'model': model_name, 'horizon': h, 'strategy': 'prob_cutoff_0.5',
                     'threshold': 0.5, **_backtest_signals(sig_pc)})

        # 2. top-5%
        T_top5 = top_k_threshold(h_val['mean'].values, k_pct=5.0)
        sig_t5 = signals_per_horizon_threshold(test_predictions, h, threshold=T_top5)
        rows.append({'model': model_name, 'horizon': h, 'strategy': 'top_5%',
                     'threshold': T_top5, **_backtest_signals(sig_t5)})

        # 3. top-10%
        T_top10 = top_k_threshold(h_val['mean'].values, k_pct=10.0)
        sig_t10 = signals_per_horizon_threshold(test_predictions, h, threshold=T_top10)
        rows.append({'model': model_name, 'horizon': h, 'strategy': 'top_10%',
                     'threshold': T_top10, **_backtest_signals(sig_t10)})

        # 4. isotonic + 0.5 на калиброванных вероятностях
        sig_iso = signals_per_horizon_threshold(test_pred_calibrated, h, threshold=0.5)
        rows.append({'model': model_name, 'horizon': h, 'strategy': 'isotonic_0.5',
                     'threshold': 0.5, **_backtest_signals(sig_iso)})

        # 5. max_pnl на val
        h_val_lr = val_lr_targets_local[val_lr_targets_local['horizon'] == h]
        merged = h_val.merge(h_val_lr, on=['timestamp', 'ticker', 'horizon'], how='inner')
        if merged.empty:
            T_pnl = 0.5
        else:
            T_pnl, _ = max_pnl_threshold(
                merged['mean'].to_numpy(), merged['actual'].to_numpy(),
                cost_per_trade=cost_per_trade, min_trades=50,
            )
        sig_pnl = signals_per_horizon_threshold(test_predictions, h, threshold=T_pnl)
        rows.append({'model': model_name, 'horizon': h, 'strategy': 'max_pnl',
                     'threshold': T_pnl, **_backtest_signals(sig_pnl)})
    return rows


results: list[dict] = []
results.extend(_evaluate_strategies(
    'iTransformer',
    val_predictions_it, test_predictions_it,
    val_targets, val_lr_targets_it,
))
results.extend(_evaluate_strategies(
    'LightGBM',
    val_predictions_gbm, test_predictions_gbm,
    val_targets, val_lr_targets_gbm,
))

results_df = pd.DataFrame(results)
print(f'{len(results_df)} strategy × horizon × model evaluations done')

## 6. Strategy comparison matrix

In [ ]:
# Sharpe pivot table
sharpe_pivot = results_df.pivot_table(
    index='strategy', columns=['model', 'horizon'],
    values='sharpe', aggfunc='first',
)
print('=== Test Sharpe matrix ===')
print(sharpe_pivot.round(3).to_string())

ret_pivot = results_df.pivot_table(
    index='strategy', columns=['model', 'horizon'],
    values='total_return', aggfunc='first',
)
print('\n=== Test total_return (%) matrix ===')
print((ret_pivot * 100).round(2).to_string())

n_buy_pivot = results_df.pivot_table(
    index='strategy', columns=['model', 'horizon'],
    values='n_buy', aggfunc='first',
)
print('\n=== n_BUY matrix ===')
print(n_buy_pivot.astype(int).to_string())

win_pivot = results_df.pivot_table(
    index='strategy', columns=['model', 'horizon'],
    values='win_rate', aggfunc='first',
)
print('\n=== win_rate matrix ===')
print(win_pivot.round(3).to_string())

## 7. VERDICT — есть ли положительный Sharpe?

In [ ]:
positive = results_df[results_df['sharpe'] > 0].sort_values('sharpe', ascending=False)
if not positive.empty:
    print('=== ✓ POSITIVE SHARPE НАЙДЕН ===')
    print(positive.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    best = positive.iloc[0]
    print(f'\nЛучшая комбинация:')
    print(f'  model={best["model"]}, strategy={best["strategy"]}, h={best["horizon"]}')
    print(f'  Sharpe={best["sharpe"]:.3f}, return={best["total_return"]*100:.2f}%, '
          f'n_BUY={int(best["n_buy"])}, win_rate={best["win_rate"]:.3f}')
    print('\nЗАКЛЮЧЕНИЕ: задача обучаема. Стратегия использования предсказаний')
    print('была главным bottleneck\'ом, а не модель. Дальнейшие архитектурные')
    print('правки (Sprint 4-5) могут улучшить эту цифру.')
else:
    print('=== ✗ ВСЕ СТРАТЕГИИ ДАЮТ ОТРИЦАТЕЛЬНЫЙ SHARPE ===')
    least_bad = results_df.sort_values('sharpe', ascending=False).iloc[0]
    print(f'Наименее плохая: {least_bad["model"]} / {least_bad["strategy"]} / h={least_bad["horizon"]}')
    print(f'  Sharpe={least_bad["sharpe"]:.3f}, return={least_bad["total_return"]*100:.2f}%')
    print('\nЗАКЛЮЧЕНИЕ: cost-aware labels слишком строгие при текущем')
    print('signal-to-noise. Архитектурные спринты НЕ ПОМОГУТ. Варианты:')
    print('  1. Loosen labels: target=(lr > 0) без cost adjustment')
    print('  2. Tighten labels: target=(lr > 3*cost) — реже UP, но крупнее')
    print('  3. Длинные горизонты: h=144 (12ч), h=288 (1д)')
    print('  4. Добавить FUTOI / multi-ticker / tick-данные (Sprint 1 видел AUC max 0.64)')

## 8. Визуализация

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, model_name in zip(axes, ['iTransformer', 'LightGBM']):
    sub = results_df[results_df['model'] == model_name]
    pivot = sub.pivot_table(index='strategy', columns='horizon', values='sharpe')
    pivot.plot.bar(ax=ax)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{model_name}: Sharpe per (strategy × horizon)')
    ax.set_ylabel('Sharpe (test)')
    ax.grid(alpha=0.3)
    ax.legend(title='horizon')
plt.tight_layout()
plt.show()

# Pred distribution: модель смещение vs prior
fig, axes = plt.subplots(1, len(cfg.data.horizons), figsize=(16, 3.5), sharey=True)
if len(cfg.data.horizons) == 1:
    axes = [axes]
for ax, h in zip(axes, cfg.data.horizons):
    it_h = test_predictions_it[test_predictions_it['horizon'] == h]['mean'].values
    gbm_h = test_predictions_gbm[test_predictions_gbm['horizon'] == h]['mean'].values
    p_up = float((test_targets[test_targets['horizon'] == h]['actual'] >= 0.5).mean())
    ax.hist(it_h, bins=40, alpha=0.5, label='iTransformer', color='steelblue')
    ax.hist(gbm_h, bins=40, alpha=0.5, label='LightGBM', color='darkorange')
    ax.axvline(p_up, ls='--', color='red', label=f'P(UP)={p_up:.3f}')
    ax.set_title(f'h={h}')
    ax.set_xlabel('predicted prob')
    ax.legend()
axes[0].set_ylabel('count')
plt.suptitle('Test prediction distribution: model vs prior')
plt.tight_layout(); plt.show()

In [ ]:
# Дополнительная диагностика: bias относительно prior на test.
print('=== Mean prediction vs actual P(UP) on test ===')
print(f'{"horizon":>8} | {"P(UP)":>7} | {"iTrans mean":>12} | {"GBM mean":>10} | '
      f'{"iTrans bias":>12} | {"GBM bias":>10}')
for h in cfg.data.horizons:
    p_up = float((test_targets[test_targets['horizon'] == h]['actual'] >= 0.5).mean())
    it_mean = float(test_predictions_it[test_predictions_it['horizon'] == h]['mean'].mean())
    gbm_mean = float(test_predictions_gbm[test_predictions_gbm['horizon'] == h]['mean'].mean())
    print(f'{h:>8} | {p_up:>7.3f} | {it_mean:>12.4f} | {gbm_mean:>10.4f} | '
          f'{(it_mean-p_up)*100:>+11.1f}pp | {(gbm_mean-p_up)*100:>+9.1f}pp')

print('\n→ LightGBM mean ближе к P(UP) — он калиброван by-design (BCE без logit-adjust).')
print('  iTransformer mean смещён вверх — побочный эффект logit_adjust_tau=0.5.')
print('  Isotonic strategy должна устранить смещение для iTransformer.')